In [1]:
# https://www.kaggle.com/competitions/orbit-wars/overview

In [2]:
!pip install --upgrade "kaggle-environments>=1.28.0" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 13.1 MB/s eta 0:00:00


In [3]:
import math, random, json, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from collections import deque
from kaggle_environments import make
import matplotlib.pyplot as plt
from collections import deque
import matplotlib.animation as animation
from tqdm import tqdm
import zlib
import gc
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, CENTER, ROTATION_RADIUS_LIMIT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

In [4]:
ID, OWNER, X, Y, RADIUS, SHIPS, PROD = 0, 1, 2, 3, 4, 5, 6
SUN_X, SUN_Y, SUN_R = 50.0, 50.0, 10.0

In [5]:
env = make("orbit_wars", debug=True)

In [6]:
class Environment:
    def __init__(self):
        self.env = make("orbit_wars", debug=False)

    def compute_reward(self, prev_obs, new_obs):
        player = new_obs.get("player", 0)
        prev_planets = prev_obs.get("planets", [])
        new_planets = new_obs.get("planets", [])
    
        total_prod = sum(p[PROD] for p in new_planets) or 1
        total_planets = len(new_planets) or 1
    
        my_prod = sum(p[PROD] for p in new_planets if p[OWNER] == player) / total_prod
        enemy_prod = sum(p[PROD] for p in new_planets if p[OWNER] != player and p[OWNER] != -1) / total_prod
        old_my_prod = sum(p[PROD] for p in prev_planets if p[OWNER] == player) / total_prod
        old_enemy_prod = sum(p[PROD] for p in prev_planets if p[OWNER] != player and p[OWNER] != -1) / total_prod
    
        delta_my_prod = my_prod - old_my_prod
        delta_enemy_prod = old_enemy_prod - enemy_prod
        map_control = my_prod - enemy_prod
    
        my_planets = sum(1 for p in new_planets if p[OWNER] == player) / total_planets
        old_my_planets = sum(1 for p in prev_planets if p[OWNER] == player) / total_planets
        delta_planets = my_planets - old_my_planets
    
        my_ships = sum(p[SHIPS] for p in new_planets if p[OWNER] == player)
        enemy_ships = sum(p[SHIPS] for p in new_planets if p[OWNER] != player and p[OWNER] != -1)
        total_ships = (my_ships + enemy_ships) or 1
        ship_advantage = (my_ships - enemy_ships) / total_ships
    
        old_my_ships = sum(p[SHIPS] for p in prev_planets if p[OWNER] == player)
        old_enemy_ships = sum(p[SHIPS] for p in prev_planets if p[OWNER] != player and p[OWNER] != -1)
        old_total_ships = (old_my_ships + old_enemy_ships) or 1
        old_ship_advantage = (old_my_ships - old_enemy_ships) / old_total_ships
        delta_ships = ship_advantage - old_ship_advantage
    
        return (
            np.sign(delta_my_prod) * np.log1p(abs(delta_my_prod) * total_prod)
            + np.sign(delta_enemy_prod) * np.log1p(abs(delta_enemy_prod) * total_prod)
            + np.sign(delta_planets) * np.log1p(abs(delta_planets) * total_planets)
            + map_control
            + ship_advantage
            + np.sign(delta_ships) * np.log1p(abs(delta_ships) * total_ships)
        )

    def __call__(self, P1, P2):
        self.env.reset()
        prev_obs_1 = self.env.state[0].observation
        prev_obs_2 = self.env.state[1].observation
    
        while True:
            obs_1 = self.env.state[0].observation
            obs_2 = self.env.state[1].observation
            
            action_1 = P1(obs_1)
            action_2 = P2(obs_2)
            
            self.env.step([action_1, action_2])
            
            new_obs_1 = self.env.state[0].observation
            new_obs_2 = self.env.state[1].observation
            
            r1 = self.compute_reward(prev_obs_1, new_obs_1)
            r2 = self.compute_reward(prev_obs_2, new_obs_2)
            
            prev_obs_1 = new_obs_1
            prev_obs_2 = new_obs_2
            
            done = self.env.state[0].status != "ACTIVE"
            
            yield r1, r2, done
            
            if done:
                break

In [7]:
def gen_features(obs):
    player = obs.get("player", 0)
    planets = sorted(obs.get("planets", []), key=lambda p: p[ID])
    fleets  = sorted(obs.get("fleets", []), key=lambda f: f[ID])
    initial_planets = obs.get("initial_planets", [])
    angular_velocity = obs.get("angular_velocity", 0.0)

    init_map = {i[ID]: i for i in initial_planets}

    features = [angular_velocity]

    enemy_planets = [p for p in planets if p[OWNER] != player and p[OWNER] != -1]

    for p in planets:
        dist_to_nearest_enemy = min(
            (math.hypot(p[X] - e[X], p[Y] - e[Y]) / 100.0 for e in enemy_planets),
            default=1.0
        )

        init = init_map.get(p[ID])
        orbital_radius = (
            math.hypot(init[X] - SUN_X, init[Y] - SUN_Y) / 100.0
            if init else 1.0
        )

        features.extend([
            float(p[OWNER] == player),
            float(p[OWNER] == -1),
            float(p[OWNER] != player and p[OWNER] != -1),
            p[X] / 100.0,
            p[Y] / 100.0,
            np.log1p(p[SHIPS]),
            p[PROD] / 5.0,
            math.hypot(p[X] - SUN_X, p[Y] - SUN_Y) / 100.0,
            orbital_radius,
            float(orbital_radius + p[RADIUS] >= (50/100)),
            dist_to_nearest_enemy,
        ])

    for f in fleets:
        features.extend([
            float(f[1] == player),
            f[2] / 100.0,
            f[3] / 100.0,
            np.log1p(f[6]),
            math.cos(f[4]),
            math.sin(f[4]),
        ])

    x = np.zeros(512, dtype=np.float32)
    l = min(len(features), 512)
    x[:l] = features[:l]

    norm = np.linalg.norm(x)
    return x / (norm + 1e-8)

In [8]:
class ActorCritic(nn.Module):
    def __init__(self, input_size=512, hidden_size=256, out=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, out),
            nn.ReLU(),
        )
        self.actor_planet = nn.Linear(out, 60)
        self.actor_angle_mean = nn.Linear(out, 1)
        self.actor_angle_std = nn.Linear(out, 1)
        self.actor_ships_alpha = nn.Linear(out, 1)
        self.actor_ships_beta = nn.Linear(out, 1)
        self.critic = nn.Linear(out, 1)
        self.actions = nn.Linear(out, 1)

    def forward(self, x):
        h = self.mlp(x)

        planet_logit = self.actor_planet(h)

        angle_mean = torch.tanh(self.actor_angle_mean(h)) * math.pi

        angle_std = F.softplus(self.actor_angle_std(h)) + 1e-4
        angle_std = torch.clamp(angle_std, 0.05, 1.0)

        ships_alpha = F.softplus(self.actor_ships_alpha(h)) + 1e-4
        ships_beta  = F.softplus(self.actor_ships_beta(h)) + 1e-4

        ships_alpha = torch.clamp(ships_alpha, 0.5, 5.0)
        ships_beta  = torch.clamp(ships_beta, 0.5, 5.0)

        value = self.critic(h)

        n_actions = F.softplus(
            self.actions(h)
        ).round().int()

        return planet_logit, angle_mean, angle_std, ships_alpha, ships_beta, n_actions, value

In [9]:
class A2CAgent:
    def __init__(self, input_size=512, hidden_size=256, lr=1e-3, gamma=0.5, entropy_coef=0.9, value_coef=0.5):
        self.model = ActorCritic(input_size, hidden_size).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=50
        )
        self.gamma   = gamma
        self.memory  = []
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef

    def save(self, path):
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path))
        
    def __call__(self, obs):
        player = obs.get("player", 0)
        planets = [Planet(*p) for p in obs.get("planets", [])]
        angular_vel = obs.get("angular_velocity", 0)
        
        owned   = [p.id for p in planets if p.owner == player]
    
        if not owned:
            return []
    
        features = gen_features(obs)
        x = torch.FloatTensor(features).unsqueeze(0).to(device)

        planet_logit, angle_mean, angle_std, ships_alpha, ships_beta, n_actions, value = self.model(x)
        number_of_actions = int(n_actions.clamp(0, len(owned)).item())
        
        available = owned.copy()
        actions = []
        
        for _ in range(number_of_actions):
            planet_logit, angle_mean, angle_std, ships_alpha, ships_beta, _, value = self.model(x)
            logits = planet_logit.squeeze(0)
        
            if not available:
                break
        
            mask = torch.full_like(logits, float("-inf"))
            mask[available] = 0.0
        
            dist_planet = Categorical(logits=logits + mask)
            source_idx = dist_planet.sample().item()
        
            if source_idx not in available:
                break
        
            available.remove(source_idx)
        
            source = planets[source_idx]
        
            dist_angle = torch.distributions.Normal(
                angle_mean.squeeze(0),
                angle_std.squeeze(0)
            )
            angle_action = dist_angle.sample()
        
            dist_frac = torch.distributions.Beta(
                ships_alpha.squeeze(0),
                ships_beta.squeeze(0)
            )
            frac_action = dist_frac.sample()
        
            ships = max(0, int(frac_action.item() * source.ships))
        
            log_prob = (
                dist_planet.log_prob(torch.tensor(source_idx).to(device))
                + dist_angle.log_prob(angle_action)
                + dist_frac.log_prob(frac_action)
            )
        
            entropy = (
                dist_planet.entropy()
                + dist_angle.entropy()
                + dist_frac.entropy()
            )
        
            self.memory.append([log_prob, value.squeeze(0), None, entropy])
        
            actions.append([source.id, angle_action.item(), ships])
        
        return actions
        
    def store_reward(self, reward):
        if self.memory:
            lp, v, _, ent = self.memory[-1]
            self.memory[-1] = (lp, v, reward, ent)

    def update(self, next_value=0.0):
        if not self.memory:
            return

        R = next_value
        returns = []
        for _, _, r, _ in reversed(self.memory):
            R = (r or 0.0) + self.gamma * R
            returns.insert(0, R)

        returns = torch.FloatTensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        actor_loss  = 0.0
        critic_loss = 0.0
        entropy_loss = 0.0

        for (log_prob, value, _, entropy), R in zip(self.memory, returns):
            advantage = R - value.item()
            actor_loss  += -log_prob * advantage
            critic_loss += (value.squeeze() - R) ** 2
            entropy_loss += -entropy

        loss = actor_loss + self.value_coef * critic_loss + self.entropy_coef * entropy_loss

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        self.memory = []

In [10]:
def random_agent(obs):
    planets = obs.get("planets", [])
    player = obs.get("player", 0)
    owned = [p for p in planets if p[OWNER] == player]
    if not owned:
        return []
    src = random.choice(owned)
    angle = random.uniform(-math.pi, math.pi)
    ships = random.randint(1, max(1, src[SHIPS]))
    
    return random.choice([[], [[src[ID], angle, ships]]])

In [11]:
def make_snapshot_agent(state_dict):
    model = ActorCritic().to(device)
    model.load_state_dict(state_dict)
    model.eval()
    def fn(obs, config=None):
        return model
        
    return fn

def save_game(env, agent, opponent, game, pool):
    path = f"orbit_agent_{game}.pt"
    agent.save(path)
    pool.append(make_snapshot_agent(copy.deepcopy(agent.model.state_dict())))

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return path


def train(n_games=100, pool_size=8, save_every=250):
    env = Environment()
    agent = A2CAgent()
    # agent.load("/kaggle/input/models/hermengardo/ow/pytorch/a2c/2/orbit_agent_5000 (1).pt")

    pool = deque(maxlen=pool_size)
    recent_rewards = deque(maxlen=50)
    recent_wins = deque(maxlen=50)
    recent_losses = deque(maxlen=50)
    pbar = tqdm(range(n_games))

    wins = 0
    losts = 0
    decay = 0.9
    path = None

    for game in pbar:
        if pool:
            opponent = random.choice(list(pool))
        else:
            pool.append(random_agent)
            opponent = random.choice(list(pool))

        episode_reward = 0.0
        for r1, r2, done in env(agent, opponent):
            agent.store_reward(r1)
            episode_reward += r1
        
        result_0 = env.env.state[0].reward or 0
        agent.store_reward(100 if result_0 == 1 else -100 if result_0 == -1 else 0)
        agent.update()
        agent.scheduler.step(episode_reward)
        agent.entropy_coef = max(0.01, agent.entropy_coef * decay)
        
        if result_0 == 1:
            wins += 1
            recent_wins.append(1)
            recent_losses.append(0)
        elif result_0 == -1:
            losts += 1
            recent_wins.append(0)
            recent_losses.append(1)

        recent_rewards.append(episode_reward)

        if game % save_every == 0 and game != 0:
            path = save_game(env, agent, opponent, game, pool)
            
        pbar.set_postfix({
            "reward": f"{episode_reward:.2f}",
            "avg_reward": f"{sum(recent_rewards) / len(recent_rewards):.2f}",
            "wr/50": f"{sum(recent_wins) / len(recent_wins):.2%}" if recent_wins else "0%",
            "lr/50": f"{sum(recent_losses) / len(recent_losses):.2%}" if recent_losses else "0%",
            "pool": len(pool),
        })

    path = save_game(env, agent, opponent, n_games, pool)
    
    return path

In [12]:
path = train(n_games=5000, pool_size=8, save_every=50)

100%|██████████| 5000/5000 [9:29:22<00:00,  6.83s/it, reward=441.07, avg_reward=540.10, wr/50=86.00%, lr/50=14.00%, pool=8]
